# 2pt N-State Fit Template

This notebook is a runnable example of the existing multi-exponential fit workflow using repository-tracked example correlator data.
Edit the input block below and call the same backend used by the CLI and plain-text input files.


## Imports / Setup


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_nstate_fit_input_text,
    run_nstate_fit_from_notebook,
    validate_nstate_notebook_config,
)


## User Inputs

The fields below mirror the current plain-text N-state fit input format.
The defaults point to realistic example data included in the repository.


In [ ]:
EXAMPLE_DATA = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "nstate_fit_notebook"

workflow_config = {
    "title_pattern": "l64c64a076_m140_fit_k0_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "c2pt": str(EXAMPLE_DATA / "c2pt_5_5_k0_pz*_real.csv"),
    "pzlist": [0],
    "fold_t": "none",
    "tsrange": [0, 24],
    "model": "normal",
    "nstates": [1, 2],
    "tmax": 12,
    "binsize": 1,
    "bootstrap_samples": 64,
    "bootstrap_size": 64,
    "seed": 2026,
    "plot": True,
    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`: Spatial lattice extent `Ns`.
- `nt`: Temporal lattice extent `Nt`.
- `lattice_spacing_fm`: Lattice spacing in fm for metadata and summaries.
- `c2pt`: Correlator CSV path or wildcard pattern. Keep `*` in the filename when using multiple `pz` values.
- `pzlist`: List of momentum indices to analyze, for example `[0]` or `[0, 1]`.
- `fold_t`: Time-folding mode before fitting.
  Choices: `"none"` or `False` = no folding; `"periodic"` or `True` = symmetric fold; `"antiperiodic"` = antisymmetric fold.
- `tsrange`: Two integers `[t_start, t_end]` selecting the raw correlator window passed into the fit workflow.
- `model`: Correlator model.
  Choices: `"normal"` = sum of exponentials; `"symmetric"` = cosh-like forward + backward form; `"antisymmetric"` = sinh-like forward - backward form.
- `nstates`: Which fits to run. Allowed values are subsets of `[1, 2, 3]`.
  Examples: `[1]`, `[1, 2]`, `[1, 2, 3]`, or `[2]` if you want to rely on cached 1-state results when available.
- `tmax`: Fit upper bound.
  Choices: integer value such as `12`, or `None` to use automatic selection from the effective-mass quality rule.
- `binsize`: Integer configuration bin size. Use `1` for no binning.
- `bootstrap_samples`: Number of bootstrap resamples. `None` lets the backend choose automatically.
- `bootstrap_size`: Number of binned configurations drawn per bootstrap sample. `None` uses the backend default.
- `seed`: Random seed for reproducible bootstrap sampling.
- `plot`: Whether to generate plots automatically.
  Choices: `True` or `False`.
- `results_dir`: Output directory. If omitted or set to `None`, outputs go to the notebook working directory.

Practical note:
- Higher-state fits are initialized hierarchically. If matching lower-state results already exist in the output directory, the code can reuse their plateau information as cache.


## Input Summary / Validation

This notebook follows the same single-config pattern as the TGEVP template.
Fields that belong to the plain-text input file are rendered below; notebook-only runtime fields such as `results_dir` stay in the same config for convenience.


In [ ]:
print(pretty_print_config(workflow_config))
print(render_nstate_fit_input_text(workflow_config))
parsed_nstate = validate_nstate_notebook_config(workflow_config)
parsed_nstate


## Run Analysis


In [ ]:
nstate_outputs = run_nstate_fit_from_notebook(workflow_config)
for path in nstate_outputs:
    print(path)


## Inspect Outputs

The fit writes tables, bootstrap samples, plots, and a plotting notebook under `examples/outputs/`.


In [ ]:
for path in nstate_outputs:
    print(Path(path).name)
